In [1]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
print("length of dataset in characters:", len(text))

length of dataset in characters: 1115394


In [3]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
stoi = { ch: i for i, ch in enumerate(chars) }
itos = { i: ch for i, ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there!"))
print(decode(encode("hii there!")))

[46, 47, 47, 1, 58, 46, 43, 56, 43, 2]
hii there!


In [10]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:1000])
print(data.shape, data.dtype)


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57,  1, 47, 57,  1, 41, 

In [11]:
n = int(0.9 * len(data)) # first 90% will be train data
train_data = data[:n] # first 90% will be train data
val_data = data[n:] # last 10% will be val data

In [12]:
block_size = 8 # context length for training
train_data[:block_size+1] # first 9 characters of train data

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [14]:
x = train_data[:block_size] # first 8 characters of train data
y = train_data[1:block_size+1] # next 8 characters of train data
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context}, target: {target}")

when input is tensor([18]), target: 47
when input is tensor([18, 47]), target: 56
when input is tensor([18, 47, 56]), target: 57
when input is tensor([18, 47, 56, 57]), target: 58
when input is tensor([18, 47, 56, 57, 58]), target: 1
when input is tensor([18, 47, 56, 57, 58,  1]), target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]), target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), target: 58


In [ ]:
torch.manual_seed(1337) # fix seed for reproducibility
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
def get_batch(split):
    """
    Generate a small batch of data of inputs x and targets y, depending on the split.
    split: 'train' or 'val'
    """
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y


xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size):  # batch dimension
    for t in range(block_size):  # time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")


In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337) # fix seed for reproducibility

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # embedding table for tokens

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # get logits from embedding table
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) # compute loss
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        """
        Generate new tokens in the language model.
        idx: input tokens
        max_new_tokens: maximum number of new tokens to generate
        """
        for _ in range(max_new_tokens):
            logits, _ = self(idx) # forward pass
            logits = logits[:, -1, :] # get the last time step logits
            probs = F.softmax(logits, dim=-1) # compute probabilities
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb) # forward pass  
print(logits.shape) # (batch_size * block_size, vocab_size)
print(loss) # loss value

idx = torch.zeros((1, 1), dtype=torch.long) # start with a single token
print(decode(model.generate(idx, max_new_tokens=100)[0].tolist())) # generate new tokens

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [28]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3) # optimizer

In [36]:
batch_size = 32 # how many independent sequences will we process in parallel?
for step in range(10000):
    xb, yb = get_batch('train') # get batch of data
    logits, loss = model(xb, yb) # forward pass
    optimizer.zero_grad(set_to_none=True) # zero gradients
    loss.backward() # backward pass
    optimizer.step() # update weights

    if step % 100 == 0:
        print(f"step {step}: loss {loss.item()}") # print loss every 10 steps
print(loss.item()) # final loss value

step 0: loss 2.5222768783569336
step 100: loss 2.5372188091278076
step 200: loss 2.5158963203430176
step 300: loss 2.50382924079895
step 400: loss 2.4178593158721924
step 500: loss 2.540128231048584
step 600: loss 2.4389612674713135
step 700: loss 2.6755776405334473
step 800: loss 2.3983147144317627
step 900: loss 2.514069080352783
step 1000: loss 2.465423822402954
step 1100: loss 2.4735515117645264
step 1200: loss 2.483755588531494
step 1300: loss 2.444798707962036
step 1400: loss 2.5974466800689697
step 1500: loss 2.4312326908111572
step 1600: loss 2.465223550796509
step 1700: loss 2.4884133338928223
step 1800: loss 2.5662145614624023
step 1900: loss 2.44449782371521
step 2000: loss 2.5472488403320312
step 2100: loss 2.4675867557525635
step 2200: loss 2.5351295471191406
step 2300: loss 2.3684194087982178
step 2400: loss 2.417296886444092
step 2500: loss 2.5745158195495605
step 2600: loss 2.4143502712249756
step 2700: loss 2.418973445892334
step 2800: loss 2.494617462158203
step 2900:

In [38]:
idx = torch.zeros((1, 1), dtype=torch.long) # start with a single token
print(decode(model.generate(idx, max_new_tokens=300)[0].tolist())) # generate new tokens


IUS: plllyss,
BOricand t me kusotenthayorieanatin inou idis ne horsercoownd
ORGunde ll mugl wrg g be, to fr t athame-he's h f we frs it tepodestha wird ad s; belesuin byes mb. Ond tand m:
Wed toshoths 'd ath! s pouthas COPrgoomur d y; withefeloulorneey mya chile.
SThyof wiorimy heig, ond watomow b,



In [ ]:
torch.cuda.is

False